In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from rake_nltk import Rake
import nltk
import matplotlib.pyplot as plt

# Loading to db
# importing os module for environment variables
import os
# importing necessary functions from dotenv library
from dotenv import load_dotenv 
# loading variables from .env file
load_dotenv() 
import psycopg2
from sqlalchemy import create_engine

In [ ]:
cf = pd.read_csv("reddit_clinical_trials_data.csv")

In [ ]:
cf['funder_type'].unique()
cf['start_year'].min()

## The two different datasets

### 1. Duplicate rows: Every paper has multiple keywords (this is the original dataset)

In [ ]:
cf_k = cf

In [ ]:
cf_k = cf_k.drop(columns=['study_status', 
                          'summary', 
                          'conditions', 
                          'pom', 
                          'sponsor',
                          'collaborators',
                          'sex', 
                          'age', 
                          'enrollment', 
                          'funder_type', 
                          'study_type', 
                          'start_date', 
                          'completion_date', 
                          'completion_year'])

In [ ]:
cf_k

In [ ]:
# To warehouse
# Connect to the database
conn = psycopg2.connect(
    dbname=os.getenv("DBNAME"),
    user=os.getenv("DBUSER"),
    password=os.getenv("DBPASSWORD"),
    port=os.getenv("DBPORT"),
    host=os.getenv("DBHOST")
)
conn_string=os.getenv("CONNSTRING")

# Create engine
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBUSER')}:{os.getenv('DBPASSWORD')}"
    f"@{os.getenv('DBHOST')}:{os.getenv('DBPORT')}/{os.getenv('DBNAME')}"
)

In [ ]:
cf_k.to_sql("clinical_trials_key", engine, if_exists="append", index=False, method="multi", chunksize=300)

### 2. Unique rows for each paper: Keywords are in a list

In [ ]:
cf.duplicated(subset='nct_number').sum()

In [ ]:
cf[['nct_number']].value_counts()

In [ ]:
cf[cf['nct_number'] == 'NCT00038467'].head(5)

In [ ]:
cf_g = cf.groupby(['nct_number'], as_index = False, sort = False, dropna = False).agg({'pom':'first', 
                                                                                       'title':'first',
                                                                                       'sponsor':'first', 
                                                                                       'collaborators':'first', 
                                                                                       'sex':'first', 
                                                                                       'age':'first', 
                                                                                       'enrollment':'first',
                                                                                       'summary':'first',
                                                                                       'funder_type':'first',
                                                                                       'study_type':'first',
                                                                                       'start_date':'first',
                                                                                       'completion_date':'first',
                                                                                       'keyword': lambda x: list(x.dropna().unique()),
                                                                                       'start_year':'first',
                                                                                       'completion_year':'first',
                                                                                       'study_status': 'first'
                                                                                       })

In [ ]:
cf_g.duplicated(subset='title').sum()

In [ ]:
cf_g[['title']].value_counts().head()

In [ ]:
cf_g[cf_g['title'] == 'A Study to Evaluate the Safety and Effectiveness of a Nasal Spray to Treat Seasonal Allergies'].head(5)

In [ ]:
cf_g.to_csv('key_clinical_trials.csv', index=False)